# Mixture of Experts (MoE)

**Paper:** Switch Transformers (Fedus et al. 2022), Mixtral (Jiang et al. 2024)

**Used in:** Mixtral-8x7B, GPT-4 (rumored), Grok-1, DeepSeek-MoE

## Dense vs Sparse models

**Dense model (standard Transformer):**
Every token goes through every parameter at every layer.
Doubling params = doubling compute.

**MoE model:**
Each layer has N expert FFN networks, but each token is routed to only K of them (typically K=2).
Doubling params ≠ doubling compute — most params are dormant per token.

```
Mixtral-8x7B:
  8 experts per layer, 2 activated per token
  Total params: ~46B  (8 experts × 7B)
  Active params per token: ~12B  (2 experts × 7B, minus shared attention)
  Compute: similar to a 12B dense model
  Quality: similar to a 70B dense model
```

## Router — the gating mechanism

A small linear layer (the **router** or **gate**) assigns a score to each expert for each token.
Top-K experts are selected. Their outputs are weighted by the router scores and summed.

## Resources

| Resource | Link |
|---|---|
| Switch Transformers paper (Fedus et al. 2022) | https://arxiv.org/abs/2101.03961 |
| Mixtral-8x7B paper (Jiang et al. 2024) | https://arxiv.org/abs/2401.04088 |
| DeepSeek-MoE (fine-grained expert splitting) | https://arxiv.org/abs/2401.06066 |
| Hugging Face MoE explainer blog | https://huggingface.co/blog/moe |
| GShard (Google, MoE at scale) | https://arxiv.org/abs/2006.16668 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Expert(nn.Module):
    """A single FFN expert — same as the FFN in a standard Transformer block."""
    def __init__(self, d_model, ffn_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.SiLU(),              # SiLU used in LLaMA/Mistral style
            nn.Linear(ffn_dim, d_model)
        )
    def forward(self, x):
        return self.net(x)


class MoELayer(nn.Module):
    """
    Mixture of Experts layer.
    Replaces the FFN in a Transformer block.
    """
    def __init__(self, d_model, ffn_dim, n_experts=8, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k     = top_k

        self.experts = nn.ModuleList([Expert(d_model, ffn_dim) for _ in range(n_experts)])
        self.router  = nn.Linear(d_model, n_experts, bias=False)   # the gating network

    def forward(self, x):
        """
        x: [batch, seq_len, d_model]
        """
        batch, seq_len, d_model = x.shape

        # Flatten to [batch*seq_len, d_model] — route each token independently
        x_flat = x.reshape(-1, d_model)
        n_tokens = x_flat.shape[0]

        # Router: score each expert for each token
        router_logits = self.router(x_flat)           # [n_tokens, n_experts]
        router_probs  = F.softmax(router_logits, dim=-1)  # [n_tokens, n_experts]

        # Select top-K experts per token
        top_k_probs, top_k_indices = router_probs.topk(self.top_k, dim=-1)
        # [n_tokens, top_k]

        # Normalize the top-K weights so they sum to 1
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        # Compute output: weighted sum of selected experts
        output = torch.zeros_like(x_flat)

        for k in range(self.top_k):
            expert_idx    = top_k_indices[:, k]    # which expert for each token
            expert_weight = top_k_probs[:, k]      # weight for this expert

            for e in range(self.n_experts):
                mask   = (expert_idx == e)          # tokens assigned to expert e
                if mask.any():
                    expert_out        = self.experts[e](x_flat[mask])   # forward through expert
                    output[mask]     += expert_weight[mask].unsqueeze(-1) * expert_out

        return output.reshape(batch, seq_len, d_model)


# Test
batch, seq, d_model = 2, 8, 128
moe = MoELayer(d_model=d_model, ffn_dim=512, n_experts=8, top_k=2)
x   = torch.randn(batch, seq, d_model)
out = moe(x)
print('MoE output shape:', out.shape)   # [2, 8, 128]

# Total params vs active params
total_params  = sum(p.numel() for p in moe.parameters())
active_params = sum(p.numel() for p in moe.experts[0].parameters()) * 2  # top_k=2 experts
print(f'Total params:    {total_params:,}')
print(f'Active per token:{active_params:,}  ({active_params/total_params*100:.0f}% of total)')

## Load Balancing — the training challenge

Without constraints, the router collapses — it always picks the same 1-2 experts.
The popular experts get better, the rest get no gradient and stagnate.

**Solution:** Add an auxiliary load balancing loss that penalizes uneven expert usage.

In [ ]:
def load_balance_loss(router_probs, expert_indices, n_experts):
    """
    Switch Transformer auxiliary loss.
    Encourages equal token distribution across experts.
    """
    n_tokens = router_probs.shape[0]

    # Fraction of tokens sent to each expert
    expert_mask = F.one_hot(expert_indices, n_experts).float()       # [n_tokens, n_experts]
    tokens_per_expert = expert_mask.mean(dim=0)                       # [n_experts] — fraction per expert

    # Average router probability for each expert
    router_per_expert = router_probs.mean(dim=0)                      # [n_experts]

    # Loss = n_experts * dot(tokens_per_expert, router_per_expert)
    # Minimized when distribution is uniform (1/n_experts for all)
    loss = n_experts * (tokens_per_expert * router_per_expert).sum()
    return loss


# Demonstrate: balanced vs collapsed routing
torch.manual_seed(42)
n_tokens, n_experts = 100, 8

# Balanced routing (ideal)
balanced_probs   = torch.softmax(torch.randn(n_tokens, n_experts), dim=-1)
balanced_indices = balanced_probs.argmax(dim=-1)

# Collapsed routing (always picks expert 0)
collapsed_probs   = torch.zeros(n_tokens, n_experts)
collapsed_probs[:, 0] = 1.0
collapsed_indices = collapsed_probs.argmax(dim=-1)

print('Balanced  aux loss:', load_balance_loss(balanced_probs,  balanced_indices,  n_experts).item())
print('Collapsed aux loss:', load_balance_loss(collapsed_probs, collapsed_indices, n_experts).item())
print('\nTokens per expert (balanced): ', (balanced_indices.bincount(minlength=n_experts) / n_tokens).tolist())
print('Tokens per expert (collapsed):', (collapsed_indices.bincount(minlength=n_experts) / n_tokens).tolist())